# CTB ProSiT reproduction in Google Colab

This notebook is the Colab version of the CTB ProSiT model handover. It loads the saved Petri net and simulation parameter bundles, demonstrates the ProSiT save/load API, runs the saved models, and compares the generated KPI tables with the frozen thesis outputs.

Run this notebook only after uploading or copying the complete `reproducibility/` directory to Google Drive.

## 0. Colab-only setup

Run the next cell only in Google Colab. It mounts Google Drive and changes into the uploaded `reproducibility` directory. If your folder is stored somewhere else, edit `REPRO_DIR`.

In [ ]:
# Colab only: mount Drive and move into the uploaded reproducibility folder.
try:
    import google.colab
except ImportError:
    print('Not running in Google Colab; skip this cell for local execution.')
else:
    from google.colab import drive
    from pathlib import Path
    import os

    drive.mount('/content/drive')

    REPRO_DIR = Path('/content/drive/MyDrive/reproducibility')
    if not (REPRO_DIR / 'requirements.txt').is_file():
        candidates = [p for p in Path('/content/drive/MyDrive').rglob('requirements.txt')
                      if p.parent.name == 'reproducibility']
        if not candidates:
            raise FileNotFoundError('Could not find the uploaded reproducibility folder in Google Drive.')
        REPRO_DIR = candidates[0].parent

    os.chdir(REPRO_DIR)
    print('Working directory:', Path.cwd())

In [ ]:
%pip install -r requirements.txt --disable-pip-version-check -q

## 1. Load Petri net and saved parameters

In [ ]:
import pickle
from datetime import datetime
from copy import deepcopy
from pathlib import Path
import random
import sys

import pandas as pd
import pm4py
from prosit import SimulatorParameters, SimulatorEngine

required = ['requirements.txt', 'reviewer_runner.py', 'models', 'expected_results', 'runtime']
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f'Notebook is not running inside the reproducibility folder. Missing: {missing}')

net, im, fm = pm4py.read_pnml('models/ctb_inductive_miner.pnml')
print(f'Petri net: {len(net.places)} places, {len(net.transitions)} transitions, {len(net.arcs)} arcs')

with open('models/params_baseline_rmg_max_concurrency_3.pkl', 'rb') as f:
    baseline = pickle.load(f)
with open('models/params_rules_only_workload_blind.pkl', 'rb') as f:
    rules_only = pickle.load(f)
with open('models/params_t22_closed.pkl', 'rb') as f:
    t22_closed = pickle.load(f)
with open('models/params_demand_plus_20pct.pkl', 'rb') as f:
    demand_plus_20 = pickle.load(f)

## 2. Inspect the model configurations

In [ ]:
for name, params in [('rules_only', rules_only), ('baseline', baseline),
                     ('t22_closed', t22_closed), ('demand_plus_20', demand_plus_20)]:
    print(f'\n--- {name} ---')
    print(f'  rules_mode:          {params.rules_mode}')
    print(f'  workload_features:   {params.use_workload_features}')
    print(f'  activities:          {list(params.act_to_resources.keys())}')
    print(f'  RMG resources:       {len(params.act_to_resources["RMG_receive"])}')
    print(f'  T22 in RMG_receive:  {"T22" in params.act_to_resources["RMG_receive"]}')
    print(f'  max_concurrency T06: {params.max_concurrency.get("T06", "n/a")}')

## 3. Run ProSiT directly

This cell uses the basic ProSiT pattern from the README: `engine = SimulatorEngine(params)` followed by `engine.apply(...)`. It is a short execution check, not the full thesis replication.

In [ ]:
random.seed(42)
t_start = datetime(2026, 4, 20, 18, 17)

engine = SimulatorEngine(deepcopy(baseline))
sim_log = engine.apply(n_traces=1000, t_start=t_start)

print(f'Simulated {sim_log["case:concept:name"].nunique()} cases, {len(sim_log)} events')
sim_log.head(10)

## 4. Quick KPI comparison across saved models

In [ ]:
quick_results = []
for name, params in [('baseline', baseline), ('t22_closed', t22_closed), ('demand_plus_20', demand_plus_20)]:
    random.seed(42)
    engine = SimulatorEngine(deepcopy(params))
    log = engine.apply(n_traces=1000, t_start=t_start)
    log['time:timestamp'] = pd.to_datetime(log['time:timestamp'])
    case_times = log.groupby('case:concept:name')['time:timestamp']
    turnaround = (case_times.max() - case_times.min()).dt.total_seconds() / 60
    quick_results.append({
        'scenario': name,
        'mean_turnaround_min': turnaround.mean(),
        'median_turnaround_min': turnaround.median(),
        'cases': log['case:concept:name'].nunique(),
    })
pd.DataFrame(quick_results)

## 5. Rules-only intermediate baseline

The rules-only model uses decision-tree rules but disables workload features and removes explicit workload-proxy attributes before discovery. This is the intermediate state between the no-rules endpoint and the rules+workload scenario baseline.

In [ ]:
import json

with open('models/rules_only_workload_blind_run_summary.json', encoding='utf-8') as f:
    rules_summary = json.load(f)

h = rules_summary['hyperparameters']
print('Rules-only workload-blind discovery settings')
print(f'  max_depth_tree:        {h["max_depth_tree"]}')
print(f'  use_workload_features: {h["use_workload_features"]}')
print(f'  workload_blind:        {h["workload_blind_attributes"]}')
print(f'  attribute_mode:        {h["attribute_mode"]}')

print(f'\nRemoved workload-proxy attributes ({len(h["workload_proxy_attributes_removed"])}):')
for attr in h['workload_proxy_attributes_removed']:
    print(' ', attr)

print('\nHeld-out conformance:')
for row in rules_summary['conformance']:
    if row['log'] == 'test':
        print(f'  fitness={row["fitness_log"]:.6f}, precision={row["precision_token"]:.4f}, generalization={row["generalization"]:.4f}')

## 6. ProSiT JSON export/import demonstration

ProSiT provides `to_json()` and `from_json()` as its portable parameter interface. For this CTB bundle, JSON export is useful for inspection, but the verified pickle files are the exact executable state because ProSiT 1.0.3 does not restore the empirical sampled arrays used by the calibration.

In [ ]:
Path('outputs').mkdir(exist_ok=True)

baseline.to_json('outputs/baseline_params.json')
print('Exported to outputs/baseline_params.json')

params_reloaded = SimulatorParameters(net, im, fm)
try:
    params_reloaded.from_json('outputs/baseline_params.json')
    print('JSON reload: success')
except Exception as e:
    print(f'JSON reload failed as expected for this calibrated CTB bundle: {type(e).__name__}: {e}')
    print('The notebook therefore uses verified pickle files for exact reproduction.')

## 7. Full thesis reproduction

This runs the exact 10-seed × 3-scenario × 17,892-case experiment and compares each generated result table with the frozen thesis outputs. On Colab this can take roughly 30 minutes.

In [ ]:
sys.path.insert(0, '.')
import reviewer_runner as rr

rr.verify_package_files()
output_dir = rr.run_saved_models(mode='full')

comparison = rr.compare_with_frozen_results(output_dir)
print(comparison.to_string(index=False))
print('\nFULL REPRODUCTION PASS — all frozen thesis scenario tables matched.')

In [ ]:
kpi = pd.read_csv(output_dir / 'scenario_kpi_summary.csv')
deltas = pd.read_csv(output_dir / 'scenario_paired_delta_summary.csv')

print('=== Thesis scenario KPIs: mean turnaround ===\n')
for scenario in ['baseline', 't22_closed', 'demand_plus_20pct']:
    row = kpi[(kpi['scenario'] == scenario) & (kpi['metric'] == 'mean_turnaround_min')].iloc[0]
    print(f'{scenario:20s}  {row["mean"]:.3f} [{row["ci95_lo"]:.3f}, {row["ci95_hi"]:.3f}] min')

print('\n=== Paired scenario deltas: mean turnaround ===\n')
for _, row in deltas[deltas['metric'] == 'mean_turnaround_min'].iterrows():
    print(f'{row["scenario"]:20s}  {row["mean_delta"]:+.3f} [{row["ci95_delta_lo"]:+.3f}, {row["ci95_delta_hi"]:+.3f}] min')